# AI-Generated Text Detection: fine-tuned mmBERT-base
Competition: [ai-generated-text-detection-hw3-ucu-ai-2026](https://www.kaggle.com/competitions/ai-generated-text-detection-hw3-ucu-ai-2026), 3 classes: `human`, `ai`, `fusion`, in English and German.

**What this notebook does**
- Fine-tunes the open-weight multilingual encoder **[`jhu-clsp/mmBERT-base`](https://huggingface.co/jhu-clsp/mmBERT-base)** (ModernBERT architecture, 1800+ languages, 8192-token context), loaded from Hugging Face.
- **Long texts**: the model trains with head + tail truncation (the first `HEAD_TOKENS` tokens plus the end of the text), because a `fusion` switch often sits late in the text. On validation it compares four inference strategies: head 512, head `MAX_LEN`, head + tail `MAX_LEN`, and sliding-window chunks with averaged probabilities. The best one is used for the test set.
- **Baseline comparison on the same split**: the 20k validation rows are built with exactly the same code and seed as the TF-IDF notebook. A TF-IDF (word + char n-grams) + LinearSVC baseline is trained on the **same training rows** as mmBERT. The notebook compares macro-F1, per-class F1, confusion matrices, scores per language/domain, and scores per text length.
- Saves `submission.csv` to `/kaggle/working` and shows a download link.

**Running on Kaggle**
- *Add Input → Competition → this competition*.
- *Settings → Accelerator → **GPU T4 x2*** (P100 also works, but slower). The notebook uses both T4s through `DataParallel`.
- *Settings → **Internet on*** to download the model (~1.2 GB). Alternatively, attach the model files as an input: any folder under `/kaggle/input` whose path contains `mmbert` and that has a `config.json`.
- Expected run time with the defaults (`TRAIN_N=20_000`, `MAX_LEN=1024`, 2 epochs) on T4 x2: roughly 1–1.5 h, about half of it validation/test inference. Lower `MAX_LEN` to go faster.

**T4 specifics**: FlashAttention 2 needs an Ampere or newer GPU, so the model loads with `attn_implementation="sdpa"`. Training uses fp16 mixed precision, since T4 and P100 have no bf16.

## Setup & config

In [ ]:
import os, re, gc, sys, time, math, random, socket, subprocess, warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink

import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

# ModernBERT needs transformers >= 4.48
import transformers
from packaging import version
if version.parse(transformers.__version__) < version.parse("4.48.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.48.0"], check=True)
    raise SystemExit("transformers was upgraded: restart the kernel (Run -> Restart) and run again.")
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
print("torch", torch.__version__, "| transformers", transformers.__version__)

# ---------------- config ----------------
QUICK = os.environ.get("QUICK", "0") == "1"    # tiny smoke-test run (set env QUICK=1)

MODEL_NAME   = "jhu-clsp/mmBERT-base"
RANDOM_STATE = 42          # same seed and split code as the TF-IDF notebook -> identical validation rows
VAL_N        = 20_000      # held-out validation rows (identical to the TF-IDF notebook)
TRAIN_N      = 20_000      # rows used to fine-tune mmBERT and to train the baseline
MAX_LEN      = 1024        # tokens per sequence, including special tokens (mmBERT supports up to 8192)
HEAD_TOKENS  = 256         # head + tail truncation: keep the first 256 tokens and fill the rest from the end
CHUNK_STRIDE = 768         # sliding-window inference: step between chunk starts (overlap = MAX_LEN - stride)
EPOCHS       = 2           # small training set -> a second pass helps
BATCH_SIZE   = 16          # total over all GPUs
GRAD_ACCUM   = 1           # effective batch = 16 -> ~2,500 optimizer steps over 2 epochs
LR           = 3e-5        # lower than for big data: more steps per sample, less overfitting risk
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EVAL_BATCH   = 32
N_EVALS      = 6           # quick checks on a 2k validation subset during training (3 per epoch)
SAVE_MODEL   = False       # save the fine-tuned model to /kaggle/working/mmbert-ft (~1.2 GB)

if QUICK:
    VAL_N, TRAIN_N, MAX_LEN, HEAD_TOKENS, CHUNK_STRIDE = 1_000, 2_000, 256, 64, 192

TEXT, LANG, DOMAIN, LABEL = "text", "lang", "domain", "label"
CLASSES = ["human", "ai", "fusion"]
LANGS = ["en-EN", "de-DE"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
if DEVICE.type == "cuda":
    print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
else:
    print("WARNING: no GPU. Set Settings -> Accelerator -> GPU T4 x2. Only QUICK mode is usable on CPU.")

@contextmanager
def timer(msg):
    t0 = time.time()
    print(f"[start] {msg} ...", flush=True)
    yield
    print(f"[done]  {msg}: {time.time() - t0:.1f}s", flush=True)

## Load data

In [ ]:
COMP = "ai-generated-text-detection-hw3-ucu-ai-2026"

def find_data_dir():
    for root in [Path("/kaggle/input"), Path("data"), Path("../data")]:
        if root.exists():
            hits = sorted(root.rglob("train.csv"))
            hits = [h for h in hits if COMP in str(h)] or hits
            if hits:
                return hits[0].parent
    raise FileNotFoundError("train.csv not found: add the competition data as an input")

DATA_DIR = find_data_dir()
with timer("read csv"):
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    sample_sub = pd.read_csv(next(iter(sorted(DATA_DIR.glob("sample_submission*.csv")))))
for df in (train, test):
    df[TEXT] = df[TEXT].fillna("").astype(str)
print(f"data dir: {DATA_DIR} | train {train.shape} | test {test.shape}")

## Splits (identical to the TF-IDF notebook)
The validation split uses the same stratification (label × lang × domain), size and seed as the TF-IDF notebook, so both notebooks score exactly the same 20k rows. mmBERT and the in-notebook baseline both train on the same `TRAIN_N` rows from the remaining pool.

In [ ]:
strat = train[LABEL] + "|" + train[LANG] + "|" + train[DOMAIN]
pool_idx, val_idx = train_test_split(train.index, test_size=VAL_N, stratify=strat, random_state=RANDOM_STATE)
pool, val = train.loc[pool_idx], train.loc[val_idx]

def strat_sample(df, n, seed=RANDOM_STATE):
    if n is None or n >= len(df):
        return df
    s = df[LABEL] + "|" + df[LANG] + "|" + df[DOMAIN]
    keep, _ = train_test_split(df.index, train_size=n, stratify=s, random_state=seed)
    return df.loc[keep]

trn = strat_sample(pool, TRAIN_N)
del pool; gc.collect()
y_trn = trn[LABEL].map(LABEL2ID).values
y_val = val[LABEL].map(LABEL2ID).values
print(f"train rows: {len(trn):,} | val rows: {len(val):,} | val id checksum: {pd.util.hash_pandas_object(val['id'], index=False).sum() % 10**8}")

## Baseline: TF-IDF + LinearSVC on the same training rows
Word 1–2-grams + `char_wb` 3–5-grams (first 2000 chars), hashed TF-IDF, LinearSVC (C=0.3). It is trained on the same `TRAIN_N` rows as mmBERT, so the comparison is on equal data. This runs on CPU and takes a few minutes.

In [ ]:
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.svm import LinearSVC

HASHERS = {
    "word": HashingVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b", n_features=2**21,
                              alternate_sign=False, norm=None, dtype=np.float32),
    "char": HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False, n_features=2**21,
                              alternate_sign=False, norm=None, dtype=np.float32),
}

def hashed(kind, texts, chunk=4000):
    texts = [t[:2000] for t in texts] if kind == "char" else list(texts)
    parts = Parallel(n_jobs=os.cpu_count())(delayed(HASHERS[kind].transform)(texts[i:i + chunk])
                                            for i in range(0, len(texts), chunk))
    return sp.vstack(parts).tocsr()

with timer("baseline features + fit"):
    tfidf = {k: TfidfTransformer(sublinear_tf=True) for k in HASHERS}
    X_trn = sp.hstack([tfidf[k].fit_transform(hashed(k, trn[TEXT])) for k in HASHERS], format="csr")
    baseline = LinearSVC(C=0.3).fit(X_trn, y_trn)
    del X_trn; gc.collect()

with timer("baseline predict val"):
    t0 = time.time()
    X_val = sp.hstack([tfidf[k].transform(hashed(k, val[TEXT])) for k in HASHERS], format="csr")
    base_val_pred = baseline.predict(X_val)
    base_val_time = time.time() - t0
    del X_val; gc.collect()
print(f"baseline val macro-F1 = {f1_score(y_val, base_val_pred, average='macro'):.4f}")

## Load mmBERT-base from Hugging Face
Loads from a Kaggle input if one contains the model files, otherwise from the Hub (needs Internet on).

In [ ]:
def internet_ok(host="huggingface.co", port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

def find_local_model():
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in sorted(root.rglob("config.json")):
            if "mmbert" in str(cfg).lower():
                return str(cfg.parent)
    return None

MODEL_SRC = find_local_model()
if MODEL_SRC is None:
    if not internet_ok():
        raise RuntimeError("No internet and no mmBERT files in /kaggle/input. Turn on Settings -> Internet "
                           "(needs a phone-verified Kaggle account) or attach the model as an input.")
    MODEL_SRC = MODEL_NAME
print("model source:", MODEL_SRC)

tokenizer = AutoTokenizer.from_pretrained(MODEL_SRC)
tokenizer.model_max_length = 10**9   # we truncate ourselves; silences length warnings

# special tokens the model expects around a sequence (e.g. [CLS] ... [SEP]), found generically
_with = tokenizer("a")["input_ids"]
_without = tokenizer("a", add_special_tokens=False)["input_ids"]
_k = next(i for i in range(len(_with)) if _with[i:i + len(_without)] == _without)
PREFIX, SUFFIX = _with[:_k], _with[_k + len(_without):]
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
BODY_LEN = MAX_LEN - len(PREFIX) - len(SUFFIX)
print(f"prefix={PREFIX} suffix={SUFFIX} pad={PAD_ID} | body tokens per sequence: {BODY_LEN}")

## Tokenize & look at text lengths
Each text is tokenized in full, without truncation. This shows how many texts exceed 512 / `MAX_LEN` tokens, which is what the long-text handling is for.

In [ ]:
def tokenize_full(texts, batch=2000):
    texts = list(texts)
    out = []
    for i in range(0, len(texts), batch):
        out.extend(tokenizer(texts[i:i + batch], add_special_tokens=False, verbose=False)["input_ids"])
    return out

with timer("tokenize train / val / test"):
    ids_trn, ids_val, ids_test = tokenize_full(trn[TEXT]), tokenize_full(val[TEXT]), tokenize_full(test[TEXT])

len_val = np.array([len(x) for x in ids_val])
lens = pd.DataFrame({"tokens": len_val, LANG: val[LANG].values, DOMAIN: val[DOMAIN].values})
print(f"validation token length: median={np.median(len_val):.0f}, mean={len_val.mean():.0f}, p95={np.percentile(len_val, 95):.0f}")
display(lens.groupby([LANG, DOMAIN])["tokens"].agg(
    median="median", p95=lambda x: x.quantile(0.95),
    over_512=lambda x: (x > 512).mean(), **{f"over_{MAX_LEN}": lambda x: (x > BODY_LEN).mean()}).round(3))

plt.figure(figsize=(9, 4))
sns.histplot(data=lens.assign(tokens=lens["tokens"].clip(upper=4000)), x="tokens", hue=LANG, bins=80, element="step")
for x, name in [(512, "512"), (BODY_LEN, f"MAX_LEN={MAX_LEN}")]:
    plt.axvline(x, ls="--", c="k"); plt.text(x, plt.ylim()[1] * 0.9, f" {name}")
plt.title("Validation text length in mmBERT tokens (clipped at 4000)"); plt.tight_layout(); plt.show()

## Long-text handling
Three ways to build a model input from a long token sequence:
- **head**: keep the first `n` tokens (standard truncation).
- **head + tail**: keep the first `HEAD_TOKENS` tokens and fill the remaining budget from the **end** of the text. The end is where a human → AI switch in `fusion` texts often shows.
- **chunks**: split the whole text into overlapping windows of `MAX_LEN` tokens (step `CHUNK_STRIDE`), classify each window, and average the class probabilities per text. The model sees the entire text, at a higher inference cost.

Training uses **head + tail** with `MAX_LEN`.

In [ ]:
def wrap(body):
    return PREFIX + list(body) + SUFFIX

def head(ids, max_len):
    return wrap(ids[: max_len - len(PREFIX) - len(SUFFIX)])

def head_tail(ids, max_len, head_tokens=HEAD_TOKENS):
    body = max_len - len(PREFIX) - len(SUFFIX)
    if len(ids) <= body:
        return wrap(ids)
    return wrap(ids[:head_tokens] + ids[len(ids) - (body - head_tokens):])

def chunks(ids, max_len=MAX_LEN, stride=CHUNK_STRIDE):
    body = max_len - len(PREFIX) - len(SUFFIX)
    if len(ids) <= body:
        return [wrap(ids)]
    starts = list(range(0, len(ids) - body + 1, stride))
    if starts[-1] + body < len(ids):
        starts.append(len(ids) - body)   # last window ends exactly at the end of the text
    return [wrap(ids[s:s + body]) for s in starts]

def pad_batch(seqs):
    L = max(len(s) for s in seqs)
    L = int(math.ceil(L / 8) * 8)          # multiple of 8 -> faster fp16 kernels
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s); mask[i, :len(s)] = 1
    return ids, mask

train_seqs = [head_tail(x, MAX_LEN) for x in ids_trn]
print(f"train sequences: {len(train_seqs):,} | truncated: {np.mean([len(x) > BODY_LEN for x in ids_trn]):.1%}")

## Model

In [ ]:
model_kwargs = dict(num_labels=len(CLASSES), id2label=dict(enumerate(CLASSES)), label2id=LABEL2ID,
                    attn_implementation="sdpa")       # FlashAttention 2 is not available on T4 / P100
try:   # reference_compile=False: no torch.compile inside ModernBERT (avoids compile issues on older GPUs)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_SRC, reference_compile=False, **model_kwargs)
except TypeError:   # transformers versions without this option
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_SRC, **model_kwargs)
model.to(DEVICE)
net = torch.nn.DataParallel(model) if N_GPU > 1 else model
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M | GPUs used: {max(N_GPU, 1)}")

## Inference helper
Sorts sequences by length so each batch has little padding, runs the model in fp16, and returns class probabilities in the original order.

In [ ]:
@torch.no_grad()
def predict_seqs(seqs, batch_size=EVAL_BATCH):
    net.eval()
    order = np.argsort([len(s) for s in seqs])[::-1]
    probs = np.zeros((len(seqs), len(CLASSES)), dtype=np.float32)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        ids, mask = pad_batch([seqs[j] for j in idx])
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = net(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        probs[idx] = F.softmax(logits.float(), dim=-1).cpu().numpy()
    return probs

def predict_docs(ids_list, strategy):
    """strategy: ('head', n) | ('head_tail', n) | ('chunks', n). Returns per-document probabilities."""
    kind, n = strategy
    if kind == "head":
        return predict_seqs([head(x, n) for x in ids_list])
    if kind == "head_tail":
        return predict_seqs([head_tail(x, n) for x in ids_list])
    if kind == "chunks":
        seqs, owner = [], []
        for d, x in enumerate(ids_list):
            for c in chunks(x, n):
                seqs.append(c); owner.append(d)
        p = predict_seqs(seqs)
        owner = np.array(owner)
        out = np.zeros((len(ids_list), len(CLASSES)), dtype=np.float32)
        np.add.at(out, owner, p)
        return out / np.bincount(owner, minlength=len(ids_list))[:, None]
    raise ValueError(kind)

## Fine-tune
AdamW with linear warmup/decay, fp16 mixed precision with a gradient scaler, gradient accumulation, and length-grouped batches (texts of similar length batched together, so less padding). It logs loss and throughput, and checks macro-F1 on a 2k validation subset a few times during training.

In [ ]:
def length_grouped_batches(lengths, batch_size, rng, mega=50):
    idx = rng.permutation(len(lengths))
    batches = []
    for i in range(0, len(idx), batch_size * mega):
        group = sorted(idx[i:i + batch_size * mega], key=lambda j: lengths[j])
        batches += [group[k:k + batch_size] for k in range(0, len(group), batch_size)]
    rng.shuffle(batches)
    return batches

no_decay = ("bias", "norm")
optimizer = torch.optim.AdamW(
    [{"params": [p for n, p in model.named_parameters() if not any(k in n.lower() for k in no_decay)], "weight_decay": WEIGHT_DECAY},
     {"params": [p for n, p in model.named_parameters() if any(k in n.lower() for k in no_decay)], "weight_decay": 0.0}],
    lr=LR)
steps_per_epoch = math.ceil(len(train_seqs) / BATCH_SIZE / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
use_amp = DEVICE.type == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

rng = np.random.default_rng(RANDOM_STATE)
mon_idx = rng.choice(len(val), size=min(2000, len(val)), replace=False)
mon_seqs = [head_tail(ids_val[i], MAX_LEN) for i in mon_idx]
eval_every = max(total_steps // N_EVALS, 1)
lengths = [len(s) for s in train_seqs]
history, step, t_start, n_tok = [], 0, time.time(), 0
print(f"optimizer steps: {total_steps:,} (batch {BATCH_SIZE} x accum {GRAD_ACCUM})")

for epoch in range(EPOCHS):
    batches = length_grouped_batches(lengths, BATCH_SIZE, rng)
    net.train()
    running, n_since = 0.0, 0
    for b_i, batch in enumerate(batches):
        ids, mask = pad_batch([train_seqs[j] for j in batch])
        labels = torch.tensor(y_trn[batch], device=DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            logits = net(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        loss = F.cross_entropy(logits.float(), labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        running += loss.item(); n_tok += int(mask.sum())
        if (b_i + 1) % GRAD_ACCUM == 0 or b_i == len(batches) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
            step += 1; n_since += 1
            if step % 50 == 0 or step == total_steps:
                el = time.time() - t_start
                print(f"epoch {epoch + 1} step {step}/{total_steps} | loss {running / n_since:.4f} | "
                      f"lr {scheduler.get_last_lr()[0]:.2e} | {n_tok / el:,.0f} tok/s | "
                      f"elapsed {el / 60:.1f} min | ETA {el / step * (total_steps - step) / 60:.1f} min", flush=True)
                history.append({"step": step, "loss": running / n_since})
                running, n_since = 0.0, 0
            if step % eval_every == 0 or step == total_steps:
                p = predict_seqs(mon_seqs).argmax(1)
                f1 = f1_score(y_val[mon_idx], p, average="macro")
                history.append({"step": step, "val2k_macro_f1": f1})
                print(f"   -> val (2k subset, head+tail) macro-F1 = {f1:.4f}", flush=True)
                net.train()
print(f"training done in {(time.time() - t_start) / 60:.1f} min")

hist = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(9, 3.5))
h_loss = hist.dropna(subset=["loss"])
ax.plot(h_loss["step"], h_loss["loss"], label="train loss")
if "val2k_macro_f1" in hist:
    ax2 = ax.twinx(); h = hist.dropna(subset=["val2k_macro_f1"])
    ax2.plot(h["step"], h["val2k_macro_f1"], "o-", c="tab:green", label="val 2k macro-F1"); ax2.set_ylabel("macro-F1")
ax.set_xlabel("optimizer step"); ax.set_ylabel("loss"); ax.set_title("Training curve"); plt.tight_layout(); plt.show()

if SAVE_MODEL:
    model.save_pretrained("/kaggle/working/mmbert-ft" if Path("/kaggle/working").exists() else "mmbert-ft")
    tokenizer.save_pretrained("/kaggle/working/mmbert-ft" if Path("/kaggle/working").exists() else "mmbert-ft")

## Validation: compare long-text strategies
The same fine-tuned model is scored on all 20k validation rows with each input strategy. The best strategy by macro-F1 is used for the test set. (Choosing it on validation adds a small optimistic bias to its score. With only 4 options, the bias is small.)

In [ ]:
STRATEGIES = {
    "head 512":                ("head", min(512, MAX_LEN)),
    f"head {MAX_LEN}":         ("head", MAX_LEN),
    f"head+tail {MAX_LEN}":    ("head_tail", MAX_LEN),
    f"chunks {MAX_LEN} (avg)": ("chunks", MAX_LEN),
}
val_probs, rows = {}, []
for name, strat_ in STRATEGIES.items():
    t0 = time.time()
    val_probs[name] = predict_docs(ids_val, strat_)
    pred = val_probs[name].argmax(1)
    rows.append({"strategy": name, "macro_f1": f1_score(y_val, pred, average="macro"),
                 "accuracy": accuracy_score(y_val, pred),
                 **{f"f1_{c}": v for c, v in zip(CLASSES, f1_score(y_val, pred, average=None, labels=range(3)))},
                 "infer_s": time.time() - t0})
    print(f"  {name:<22} macro-F1={rows[-1]['macro_f1']:.4f} ({rows[-1]['infer_s']:.0f}s)", flush=True)
strat_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(strat_df.style.background_gradient(subset=["macro_f1", "f1_human", "f1_ai", "f1_fusion"], cmap="Greens").format(precision=4))

BEST_STRATEGY = strat_df.loc[0, "strategy"]
mm_val_pred = val_probs[BEST_STRATEGY].argmax(1)
print(f"best strategy: {BEST_STRATEGY}")

## mmBERT per-class metrics and confusion matrix

In [ ]:
print(f"mmBERT ({BEST_STRATEGY}) validation macro-F1 = {f1_score(y_val, mm_val_pred, average='macro'):.4f}\n")
print(classification_report(y_val, mm_val_pred, target_names=CLASSES, digits=4))

## Comparison with the TF-IDF baseline (same validation rows)

In [ ]:
def summarize(name, pred, seconds):
    r = {"model": name, "macro_f1": f1_score(y_val, pred, average="macro"), "accuracy": accuracy_score(y_val, pred)}
    r.update({f"f1_{c}": v for c, v in zip(CLASSES, f1_score(y_val, pred, average=None, labels=range(3)))})
    for lang in LANGS:
        m = (val[LANG] == lang).values
        r[f"macro_f1_{lang}"] = f1_score(y_val[m], pred[m], average="macro")
    r["val_infer_s"] = seconds
    return r

cmp = pd.DataFrame([summarize(f"TF-IDF + LinearSVC ({len(trn):,} rows)", base_val_pred, base_val_time)] +
                   [summarize(f"mmBERT-base, {k}", val_probs[k].argmax(1), strat_df.set_index('strategy').loc[k, 'infer_s'])
                    for k in STRATEGIES])
display(cmp.style.background_gradient(subset=[c for c in cmp.columns if c.startswith(("macro", "f1_", "acc"))], cmap="Greens")
        .format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, pred) in zip(axes, [("TF-IDF + LinearSVC", base_val_pred), (f"mmBERT ({BEST_STRATEGY})", mm_val_pred)]):
    cm = confusion_matrix(y_val, pred, labels=range(3))
    sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f"{name}\nmacro-F1={f1_score(y_val, pred, average='macro'):.4f}"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.suptitle("Row-normalised confusion matrices, same validation rows"); plt.tight_layout(); plt.show()

cm = confusion_matrix(y_val, mm_val_pred, labels=range(3))
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"mmBERT ({BEST_STRATEGY}): counts"); plt.xlabel("predicted"); plt.ylabel("true"); plt.tight_layout(); plt.show()

### Per language × domain, and per text length
The length buckets show whether the long-text handling pays off: the baseline sees all words (char n-grams only the first 2000 characters), while a truncating transformer does not.

In [ ]:
v = val.assign(base=base_val_pred, mm=mm_val_pred, y=y_val, tokens=len_val, pos=np.arange(len(val)))
rows = []
for (lang, dom), g in v.groupby([LANG, DOMAIN]):
    rows.append({"lang": lang, "domain": dom, "n": len(g),
                 "tfidf_macro_f1": f1_score(g["y"], g["base"], average="macro"),
                 "mmbert_macro_f1": f1_score(g["y"], g["mm"], average="macro")})
dom_cmp = pd.DataFrame(rows).assign(delta=lambda d: d["mmbert_macro_f1"] - d["tfidf_macro_f1"]).sort_values("delta")
display(dom_cmp.style.background_gradient(subset=["delta"], cmap="RdYlGn", vmin=-0.1, vmax=0.1).format(precision=3))

v["length_bucket"] = pd.cut(v["tokens"], [0, 512, BODY_LEN, 2 * BODY_LEN, np.inf],
                            labels=["<=512", f"513-{BODY_LEN}", f"{BODY_LEN + 1}-{2 * BODY_LEN}", f">{2 * BODY_LEN}"])
len_rows = []
for b, g in v.groupby("length_bucket", observed=True):
    r = {"tokens": b, "n": len(g), "TF-IDF": f1_score(g["y"], g["base"], average="macro")}
    for k in STRATEGIES:
        r[k] = f1_score(g["y"], val_probs[k][g["pos"].values].argmax(1), average="macro")
    len_rows.append(r)
len_cmp = pd.DataFrame(len_rows).set_index("tokens")
display(len_cmp.style.background_gradient(axis=1, subset=[c for c in len_cmp.columns if c != "n"], cmap="Greens").format(precision=4))
len_cmp.drop(columns="n").plot.bar(rot=0, figsize=(10, 4), ylim=(0, 1), title="Macro-F1 by text length (validation)")
plt.ylabel("macro-F1"); plt.legend(loc="lower left", fontsize=8); plt.tight_layout(); plt.show()

## Predict test & save submission
Uses the best long-text strategy from validation. Saves `/kaggle/working/submission.csv` (format `id,label`, in `sample_submission.csv` order) and shows a download link. On Kaggle, the file also appears under *Output*.

In [ ]:
with timer(f"predict test ({BEST_STRATEGY})"):
    test_probs = predict_docs(ids_test, STRATEGIES[BEST_STRATEGY])
test_pred = np.array(CLASSES)[test_probs.argmax(1)]

sub = sample_sub[["id"]].merge(pd.DataFrame({"id": test["id"].values, "label": test_pred}), on="id", how="left")
assert sub["label"].notna().all() and len(sub) == len(sample_sub) and set(sub["label"]) <= set(CLASSES)

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SUB_PATH = OUT_DIR / "submission.csv"
sub.to_csv(SUB_PATH, index=False)
print(f"saved {SUB_PATH.resolve()} {sub.shape}")
display(sub.head())
display(pd.concat({"train share": train[LABEL].value_counts(normalize=True),
                   "test pred share": sub["label"].value_counts(normalize=True)}, axis=1).loc[CLASSES].round(3))

os.chdir(OUT_DIR)            # FileLink paths are relative to the working directory
display(FileLink(SUB_PATH.name))

## Summary
- **Model**: `jhu-clsp/mmBERT-base`, fine-tuned for `EPOCHS` epochs on `TRAIN_N` (20k) rows with head + tail truncation at `MAX_LEN` tokens.
- **Long texts**: four inference strategies compared on validation (head 512, head `MAX_LEN`, head + tail, sliding-window chunks). The by-length table shows where truncation hurts.
- **Baseline**: TF-IDF + LinearSVC trained on the same rows and scored on the same 20k validation rows as the TF-IDF notebook. See the comparison table, confusion matrices and per-domain deltas.
- **Output**: `submission.csv` in `/kaggle/working`, with a download link above.